# Importações

In [2]:
import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import re

# RF01 – Criar ou Carregar o Dataset de Vendas


In [5]:
def gerar_dataset_vendas(n_registros=150, seed=42):
    """
    Gera um dataset sintético de vendas com dados propositalmente sujos,
    incluindo valores nulos, strings sujas, datas inválidas e outliers.
    """

    random.seed(seed)
    np.random.seed(seed)

    produtos = [
        'Notebook',
        'Smartphone',
        'Tablet',
        'Monitor',
        'Teclado',
        'Mouse'
    ]

    precos = {
        'Notebook': 3500,
        'Smartphone': 2200,
        'Tablet': 1800,
        'Monitor': 1200,
        'Teclado': 250,
        'Mouse': 120
    }

    categorias = {
        "Notebook": "Computadores",
        "Smartphone": "Celulares",
        "Tablet": "Celulares",
        "Monitor": "Computadores",
        "Teclado": "Periféricos",
        "Mouse": "Periféricos"
    }

    regioes = [
        "Sudeste",
        "Sul",
        "Nordeste",
        "Centro-Oeste",
        "Norte"
    ]

    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]

    data_inicio = datetime(2024, 1, 1)

    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]

        data = data_inicio + timedelta(days=random.randint(0, 364))

        # Inserindo dados intencionalmente sujos para limpeza
        if random.random() < 0.05:
            quantidade = None  # valor nulo

        if random.random() < 0.04:
            preco = None  # valor nulo

        if random.random() < 0.03:
            produto = " " + produto  # espaço extra (string suja)

        data_str = (
            data.strftime("%Y-%m-%d")
            if random.random() > 0.02
            else "DATA INVALIDA"
        )

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })

    return pd.DataFrame(dados)


# Gerar e salvar
df_bruto = gerar_dataset_vendas()
os.makedirs("../data/raw", exist_ok=True)
df_bruto.to_csv("../data/raw/vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")


Dataset gerado com 150 registros.


In [6]:
df_bruto

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-01-13,Cliente_024,Mouse,Periféricos,Norte,2.0,120.0
1,2,2024-08-04,Cliente_018,Notebook,Computadores,Sul,NaN,3500.0
2,3,DATA INVALIDA,Cliente_026,Mouse,Periféricos,Sul,9.0,120.0
3,4,2024-06-23,Cliente_013,Mouse,Periféricos,Sudeste,7.0,120.0
4,5,2024-11-05,Cliente_030,Tablet,Celulares,Centro-Oeste,6.0,1800.0
...,...,...,...,...,...,...,...,...
145,146,2024-08-09,Cliente_019,Notebook,Computadores,Norte,2.0,3500.0
146,147,2024-12-09,Cliente_028,Teclado,Periféricos,Sudeste,6.0,250.0
147,148,2024-06-08,Cliente_008,Tablet,Celulares,Sudeste,10.0,1800.0
148,149,2024-07-07,Cliente_018,Tablet,Celulares,Norte,9.0,1800.0


RF02 – Inspecionar e Descrever os Dados

In [7]:
df = df_bruto

In [8]:
def inspecionar_dados(df):
  """Exibe informações básicas do df."""
  print("\n=== INSPEÇÃO INICIAL DO DATASET ===" )
  print(f"Shape: {df.shape}")
  print(f"\nColunas: {list(df.columns)}" )
  print(f"\nTipos de dados:\n{df.dtypes}" )
  print(f"\nValores nulos por coluna:\n{df.isnull().sum()}" )
  print(f"\nValores duplicados: {df.duplicated().sum()}")
  print(f"\nPrimeiros registros:\n{df.head()}" )
  print(f"\nÚltimos registros:\n{df.tail()}" )
  print(f"\nEstatísticas descritivas:\n{df.describe()}" )
  return df.describe(include="all")

inspecionar_dados(df)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (150, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        5
preco_unitario    2
dtype: int64

Valores duplicados: 0

Primeiros registros:
   id_venda     data_venda      cliente   produto     categoria        regiao  \
0         1     2024-01-13  Cliente_024     Mouse   Periféricos         Norte   
1         2     2024-08-04  Cliente_018  Notebook  Computadores           Sul   
2         3  DATA INVALIDA  Cliente_026     Mouse   Periféricos           Sul   
3         4     2024-06-2

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
count,150.000000,150,150,150,150,150,145.000000,148.000000
unique,NaN,117,30,8,3,5,NaN,NaN
top,NaN,DATA INVALIDA,Cliente_018,Mouse,Celulares,Sudeste,NaN,NaN
freq,NaN,4,8,28,51,41,NaN,NaN
mean,75.500000,NaN,NaN,NaN,NaN,NaN,5.468966,1558.513514
std,43.445368,NaN,NaN,NaN,NaN,NaN,2.808853,1190.199414
min,1.000000,NaN,NaN,NaN,NaN,NaN,1.000000,120.000000
25%,38.250000,NaN,NaN,NaN,NaN,NaN,3.000000,250.000000
50%,75.500000,NaN,NaN,NaN,NaN,NaN,5.000000,1800.000000
75%,112.750000,NaN,NaN,NaN,NaN,NaN,8.000000,2200.000000


1. O dataset possui 150 registros e 8 colunas;
2. Há colunas numéricas ('id_venda', 'quantidade', 'preco_unitario') e categóricas ('cliente', 'produto', 'categoria', 'regiao');
3. A coluna 'data_venda' está como object TEXTO e precisará ser convertida para datetime;
4. Foram encontrados valores nulos para as colunas 'quantidade' e 'preco_unitario'. Logo, precisarão ser tratados;
5. Não parece ter outliers significativos;
6. Não há registros duplicados.

In [9]:
def inspecionar_dados_vendas(df):
    print("\n=== ANÁLISE ESPECÍFICA DO DATASET ===")

    print("\nColunas categóricas:")
    #print(df["cliente"].value_counts())
    print(f"{df["produto"].value_counts()}")
    print(f"\n{df["categoria"].value_counts()}")
    print(f"\n{df["regiao"].value_counts()}")

    print("\nDatas inválidas:")
    print(df[df["data_venda"] == "DATA INVALIDA"])

    print("\nProdutos com espaços extras:")
    print(df[df["produto"] != df["produto"].str.strip()])

inspecionar_dados_vendas(df)


=== ANÁLISE ESPECÍFICA DO DATASET ===

Colunas categóricas:
produto
Mouse          28
Notebook       27
Tablet         26
Smartphone     23
Monitor        23
Teclado        21
 Tablet         1
 Smartphone     1
Name: count, dtype: int64

categoria
Celulares       51
Computadores    50
Periféricos     49
Name: count, dtype: int64

regiao
Sudeste         41
Norte           38
Nordeste        26
Sul             23
Centro-Oeste    22
Name: count, dtype: int64

Datas inválidas:
     id_venda     data_venda      cliente     produto     categoria  \
2           3  DATA INVALIDA  Cliente_026       Mouse   Periféricos   
16         17  DATA INVALIDA  Cliente_011  Smartphone     Celulares   
21         22  DATA INVALIDA  Cliente_023  Smartphone     Celulares   
106       107  DATA INVALIDA  Cliente_024    Notebook  Computadores   

           regiao  quantidade  preco_unitario  
2             Sul         9.0           120.0  
16   Centro-Oeste         6.0          2200.0  
21        Sudeste   

1. Há datas inválidas: DATA INVALIDA;
2. Coluna 'produto' possui valores com espaços em branco.

In [10]:
df

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-01-13,Cliente_024,Mouse,Periféricos,Norte,2.0,120.0
1,2,2024-08-04,Cliente_018,Notebook,Computadores,Sul,NaN,3500.0
2,3,DATA INVALIDA,Cliente_026,Mouse,Periféricos,Sul,9.0,120.0
3,4,2024-06-23,Cliente_013,Mouse,Periféricos,Sudeste,7.0,120.0
4,5,2024-11-05,Cliente_030,Tablet,Celulares,Centro-Oeste,6.0,1800.0
...,...,...,...,...,...,...,...,...
145,146,2024-08-09,Cliente_019,Notebook,Computadores,Norte,2.0,3500.0
146,147,2024-12-09,Cliente_028,Teclado,Periféricos,Sudeste,6.0,250.0
147,148,2024-06-08,Cliente_008,Tablet,Celulares,Sudeste,10.0,1800.0
148,149,2024-07-07,Cliente_018,Tablet,Celulares,Norte,9.0,1800.0


Será necessário:

1. Tratar valores nulos para as colunas 'quantidade' e 'preco_unitario';
2. Remover linhas com datas inválidas: DATA INVÁLIDA;
3. Remover espaços em branco das strings para a coluna 'produto';
4. Converter 'data_venda' para o tipo datetime.